# A2A v1.5 full Tier A → Tier B campaign
Upload this notebook to Google Colab and select a GPU runtime. It runs the frozen 30-replica, 3 μs campaign with 1 ns checkpoints. Tier B is cryptographically and logically locked until both Tier A controls pass in at least two of three replicas. Re-run the notebook after a Colab timeout; the active replica resumes from Google Drive.

## Required Drive layout
Before production, place the ten accepted system-bundle folders under `MyDrive/a2a_md_v15/bundles/`. Each folder must contain `bundle_manifest.json`, `system.xml`, `topology.pdb`, `equilibrated_state.xml`, and `native_contacts.json`. The repository auditor creates and hashes these manifests. Preliminary builder PDB files are deliberately rejected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q openmm==8.6.0 mdtraj==1.11.0 scipy==1.16.2

In [ ]:
import openmm
platforms = [openmm.Platform.getPlatform(i).getName() for i in range(openmm.Platform.getNumPlatforms())]
print('OpenMM', openmm.version.short_version, 'platforms:', platforms)
assert 'CUDA' in platforms or 'OpenCL' in platforms, 'Choose Runtime > Change runtime type > GPU.'
RUN_PLATFORM = 'CUDA' if 'CUDA' in platforms else 'OpenCL'

In [ ]:
from pathlib import Path
import json, subprocess, sys
REPO = Path('/content/ECMO-Research-Project')
if not REPO.exists():
    subprocess.run(['git','clone','https://github.com/tharranb29-spec/ECMO-Research-Project.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
BUNDLES = Path('/content/drive/MyDrive/a2a_md_v15/bundles')
RUNS = Path('/content/drive/MyDrive/a2a_md_v15/runs')
REPORTS = Path('/content/drive/MyDrive/a2a_md_v15/reports')
REPORTS.mkdir(parents=True, exist_ok=True)
MAX_REPLICA_LAUNCHES_THIS_SESSION = 1  # safest for Colab; rerun to resume/continue

In [ ]:
config = json.loads((REPO/'track3_a2a/config/md_validation.v1.5.json').read_text())
tier_a = [row['system_id'] for row in config['tiers']['tier_a_controls']]
tb = config['tiers']['tier_b_blinded_candidates']
tier_b = [f'{candidate}_{pdb}' for candidate in tb['candidate_ids'] for pdb in tb['receptor_structures']]
def verify_bundle(system_id, tier):
    path = BUNDLES/system_id/'bundle_manifest.json'
    assert path.is_file(), f'Missing bundle manifest: {path}'
    manifest = json.loads(path.read_text())
    assert manifest['system_id'] == system_id and manifest['tier'] == tier
    assert manifest['status'] == f"accepted_for_tier_{tier.lower()}_production", manifest.get('blockers')
    return manifest
for system_id in tier_a: verify_bundle(system_id, 'A')
print('Both Tier A bundles passed computational audit.')

In [ ]:
def complete(system_id, replica):
    path = RUNS/system_id/f'replica_{replica}'/'run_status.json'
    return path.is_file() and json.loads(path.read_text()).get('status') == 'complete'
def run_replica(tier, system_id, replica):
    script = REPO/'track3_a2a'/f'run_tier_{tier.lower()}_openmm_v15.py'
    command = [sys.executable,str(script),'--system-id',system_id,'--replica',str(replica),
               '--bundle',str(BUNDLES/system_id),'--output-root',str(RUNS),'--platform',RUN_PLATFORM]
    if tier == 'B': command += ['--control-gate',str(REPORTS/'control_gate_report.json')]
    subprocess.run(command,check=True)
def pending(systems):
    return [(system_id,replica) for system_id in systems for replica in (1,2,3) if not complete(system_id,replica)]

## Tier A controls
This cell launches or resumes the first incomplete control replica through `run_tier_a_openmm_v15.py`. A Colab disconnect does not invalidate it because the runner writes a durable checkpoint every 1 ns. Repeat the notebook until all six control replicas are complete. Tier B later uses the separate `run_tier_b_openmm_v15.py` guard.

In [ ]:
queue = pending(tier_a)
print('Pending Tier A replicas:', queue)
for system_id, replica in queue[:MAX_REPLICA_LAUNCHES_THIS_SESSION]:
    run_replica('A', system_id, replica)
print('Tier A complete:', not pending(tier_a))

In [ ]:
CONTROL_GATE = REPORTS/'control_gate_report.json'
if not pending(tier_a):
    subprocess.run([sys.executable,str(REPO/'track3_a2a/analyze_md_campaign_v15.py'),
                    '--tier','A','--bundles-root',str(BUNDLES),'--runs-root',str(RUNS),
                    '--output',str(CONTROL_GATE)],check=True)
gate = json.loads(CONTROL_GATE.read_text()) if CONTROL_GATE.is_file() else {}
print('Control gate:', gate.get('status','Tier A incomplete'))

## Tier B blinded candidates
The next cell cannot run unless the control report says both Tier A controls passed. It never loads potency or functional labels. There are 24 Tier B replicas; repeat the notebook as needed.

In [ ]:
if gate.get('tier_b_unlocked') is True:
    for system_id in tier_b: verify_bundle(system_id, 'B')
    queue = pending(tier_b)
    print('Pending Tier B replicas:', queue)
    for system_id, replica in queue[:MAX_REPLICA_LAUNCHES_THIS_SESSION]:
        run_replica('B', system_id, replica)
    if not pending(tier_b):
        subprocess.run([sys.executable,str(REPO/'track3_a2a/analyze_md_campaign_v15.py'),
                        '--tier','B','--bundles-root',str(BUNDLES),'--runs-root',str(RUNS),
                        '--output',str(REPORTS/'tier_b_blinded_report.json')],check=True)
else:
    print('Tier B remains locked. Finish Tier A or diagnose the failed control gate.')

In [ ]:
summary = {'tier_a_complete': not pending(tier_a), 'control_gate': gate.get('status'),
           'tier_b_unlocked': gate.get('tier_b_unlocked',False),
           'tier_b_complete': gate.get('tier_b_unlocked') is True and not pending(tier_b),
           'sampling_plan_microseconds': 3.0, 'candidate_labels_loaded': False}
print(json.dumps(summary,indent=2))